# Open Task Data Preparation

## Imports

In [1]:
import os

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from transformers import AutoTokenizer, AutoModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", DEVICE)

C:\Users\iamgr\Desktop\SC4020-Lab2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using: cuda


## Config

In [2]:
PROJECT_ROOT = os.path.abspath("..")

# Teammate inputs
SOURCE_DIR = os.path.join(PROJECT_ROOT, "data")
SOURCE_SPLIT_PATTERN = "{dataset}_{split}.csv"
SMILES_COLUMN = "compound_iso_smiles"
SEQUENCE_COLUMN = "target_sequence"
AFFINITY_COLUMN = "affinity"
DATASETS = ("davis", "kiba")
SPLITS = ("train", "val", "test")

# Stage flags
CREATE_MAPPING = True
CREATE_EMBEDDING = True
CREATE_SPLITS = True
CREATE_HOLDOUTS = True

# Outputs
OUT_DIR = os.path.join(PROJECT_ROOT, "open_task", "data")
DRUG_MAPPING_FILE = os.path.join(OUT_DIR, "drug_mapping.csv")
PROTEIN_MAPPING_FILE = os.path.join(OUT_DIR, "protein_mapping.csv")
DRUG_EMBEDDING_FILE = os.path.join(OUT_DIR, "drug_embedding.npy")
PROTEIN_EMBEDDING_FILE = os.path.join(OUT_DIR, "protein_embedding.npy")
OUT_SPLIT_PATTERN = "{dataset}_{split}.csv"
DRUG_ID_COLUMN = "drug_id"
PROTEIN_ID_COLUMN = "protein_id"

# Cold-target holdouts: proteins dropped from training only, val and test left intact
HOLDOUT_RATIOS = (0.0, 0.1, 0.2, 0.3, 0.5, 0.75)
HOLDOUT_DIR_PATTERN = os.path.join(OUT_DIR, "protein-holdout-{ratio}")
HOLDOUT_SEED = 42

# Pretrained models, used as-is
DRUG_MODEL = "DeepChem/ChemBERTa-100M-MLM"
PROTEIN_MODEL = "facebook/esm2_t6_8M_UR50D"
DRUG_BATCH_SIZE = 64
PROTEIN_BATCH_SIZE = 8
DRUG_MAX_LENGTH = 512
PROTEIN_MAX_LENGTH = None

os.makedirs(OUT_DIR, exist_ok=True)

## Source Splits

In [3]:
# keep_default_na=False so a SMILES like NA stays a string
splits = {
    (dataset, split): pd.read_csv(
        os.path.join(SOURCE_DIR, SOURCE_SPLIT_PATTERN.format(dataset=dataset, split=split)),
        keep_default_na=False,
    )
    for dataset in DATASETS
    for split in SPLITS
}

for (dataset, split), frame in splits.items():
    print(f"{dataset} {split}: {len(frame)}")

davis train: 18040
davis val: 3865
davis test: 3867
kiba train: 82359
kiba val: 17648
kiba test: 17650


## Mapping

In [4]:
if CREATE_MAPPING:
    drugs = sorted({value for frame in splits.values() for value in frame[SMILES_COLUMN]})
    proteins = sorted({value for frame in splits.values() for value in frame[SEQUENCE_COLUMN]})

    drug_mapping = pd.DataFrame({DRUG_ID_COLUMN: range(len(drugs)), SMILES_COLUMN: drugs})
    protein_mapping = pd.DataFrame({PROTEIN_ID_COLUMN: range(len(proteins)), SEQUENCE_COLUMN: proteins})

    drug_mapping.to_csv(DRUG_MAPPING_FILE, index=False)
    protein_mapping.to_csv(PROTEIN_MAPPING_FILE, index=False)
else:
    drug_mapping = pd.read_csv(DRUG_MAPPING_FILE, keep_default_na=False)
    protein_mapping = pd.read_csv(PROTEIN_MAPPING_FILE, keep_default_na=False)

print("drugs:   ", len(drug_mapping))
print("proteins:", len(protein_mapping))

drugs:    2127
proteins: 429


## Embedding

In [5]:
def embed_sequences(sequences, model_name, batch_size, max_length):
    """Return one vector per sequence, as an array of shape (len(sequences), dim)."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE)
    model.eval()

    all_vectors = []

    for start in range(0, len(sequences), batch_size):
        batch = sequences[start:start + batch_size]

        tokens = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(DEVICE)

        with torch.no_grad():
            token_vectors = model(**tokens).last_hidden_state

        # Average over tokens, ignoring padding
        mask = tokens["attention_mask"].unsqueeze(-1)
        averaged = (token_vectors * mask).sum(dim=1) / mask.sum(dim=1)

        all_vectors.append(averaged.cpu().numpy())
        print(f"  {start + len(batch)} / {len(sequences)}", end="\r")

    print()
    return np.vstack(all_vectors).astype(np.float32)

In [6]:
if CREATE_EMBEDDING:
    # Mapping order, so row i is the vector for ID i
    print("Embedding drugs with", DRUG_MODEL)
    drug_vectors = embed_sequences(
        drug_mapping[SMILES_COLUMN].tolist(), DRUG_MODEL, DRUG_BATCH_SIZE, DRUG_MAX_LENGTH)
    np.save(DRUG_EMBEDDING_FILE, drug_vectors)

    print("Embedding proteins with", PROTEIN_MODEL)
    protein_vectors = embed_sequences(
        protein_mapping[SEQUENCE_COLUMN].tolist(), PROTEIN_MODEL, PROTEIN_BATCH_SIZE, PROTEIN_MAX_LENGTH)
    np.save(PROTEIN_EMBEDDING_FILE, protein_vectors)

    print("drug vectors:   ", drug_vectors.shape)
    print("protein vectors:", protein_vectors.shape)

Embedding drugs with DeepChem/ChemBERTa-100M-MLM


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1700.91it/s]
[transformers] RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-100M-MLM
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  2127 / 2127
Embedding proteins with facebook/esm2_t6_8M_UR50D


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 8402.61it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  429 / 429
drug vectors:    (2127, 768)
protein vectors: (429, 320)


## Indexed Splits

In [7]:
if CREATE_SPLITS:
    drug_ids = dict(zip(drug_mapping[SMILES_COLUMN], drug_mapping[DRUG_ID_COLUMN]))
    protein_ids = dict(zip(protein_mapping[SEQUENCE_COLUMN], protein_mapping[PROTEIN_ID_COLUMN]))

    for (dataset, split), frame in splits.items():
        indexed = pd.DataFrame({
            DRUG_ID_COLUMN: frame[SMILES_COLUMN].map(drug_ids),
            PROTEIN_ID_COLUMN: frame[SEQUENCE_COLUMN].map(protein_ids),
            AFFINITY_COLUMN: frame[AFFINITY_COLUMN],
        })
        path = os.path.join(OUT_DIR, OUT_SPLIT_PATTERN.format(dataset=dataset, split=split))
        indexed.to_csv(path, index=False)
        print(f"{dataset} {split}: {len(indexed)} -> {os.path.basename(path)}")

davis train: 18040 -> davis_train.csv
davis val: 3865 -> davis_val.csv
davis test: 3867 -> davis_test.csv
kiba train: 82359 -> kiba_train.csv
kiba val: 17648 -> kiba_val.csv
kiba test: 17650 -> kiba_test.csv


## Protein Holdouts

In [8]:
if CREATE_HOLDOUTS:
    indexed = {
        (dataset, split): pd.read_csv(
            os.path.join(OUT_DIR, OUT_SPLIT_PATTERN.format(dataset=dataset, split=split)))
        for dataset in DATASETS
        for split in SPLITS
    }

    # One permutation per dataset, so the 0.1 holdout is a subset of 0.2, and 0.2 of 0.3
    shuffled = {
        dataset: np.random.default_rng(HOLDOUT_SEED).permutation(
            np.sort(indexed[dataset, "train"][PROTEIN_ID_COLUMN].unique()))
        for dataset in DATASETS
    }

    holdouts = []

    for ratio in HOLDOUT_RATIOS:
        directory = HOLDOUT_DIR_PATTERN.format(ratio=ratio)
        os.makedirs(directory, exist_ok=True)

        for dataset in DATASETS:
            proteins = shuffled[dataset]
            held_out = set(proteins[:round(ratio * len(proteins))])
            kept = ~indexed[dataset, "train"][PROTEIN_ID_COLUMN].isin(held_out)

            for split in SPLITS:
                frame = indexed[dataset, split]
                # Held-out proteins leave training only, so val and test stay comparable across ratios
                if split == "train":
                    frame = frame[kept]

                frame.to_csv(
                    os.path.join(directory, OUT_SPLIT_PATTERN.format(dataset=dataset, split=split)),
                    index=False)

            cold = {
                split: int(indexed[dataset, split][PROTEIN_ID_COLUMN].isin(held_out).sum())
                for split in ("val", "test")
            }
            holdouts.append({
                "ratio": ratio,
                "dataset": dataset,
                "proteins": len(proteins),
                "held out": len(held_out),
                "train": int(kept.sum()),
                "cold val rows": cold["val"],
                "cold test rows": cold["test"],
            })

    display(pd.DataFrame(holdouts).set_index(["ratio", "dataset"]))

proteins  held out  train  cold val rows  cold test rows
ratio dataset                                                          
0.00  davis         379         0  18040              0               0
      kiba          229         0  82359              0               0
0.10  davis         379        38  16225            380             389
      kiba          229        23  74013           1766            1849
0.20  davis         379        76  14434            766             796
      kiba          229        46  64867           3711            3854
0.30  davis         379       114  12640           1139            1213
      kiba          229        69  57698           5196            5359
0.50  davis         379       190   9021           1933            1968
      kiba          229       114  43502           8346            8412
0.75  davis         379       284   4511           2870            2913
      kiba          229       172  22311          12887           12979